# 3. Connect your own layout algorithm

## Before running

Use a source checkout. From its root, prepare a Python environment and install:

```sh
python -m pip install -e '.[notebooks]'
python scripts/bootstrap.py --jobs 4
python scripts/build_liblsqecc_adapter.py --source .deps/liblsqecc --build .deps/liblsqecc/build --out .deps/adapter
jupyter lab notebooks
```

The native build requires Git, CMake, a C++20 compiler and liblsqecc development
libraries; see [the integration guide](../docs/integration.md). The adapter is our
executable around liblsqecc that accepts custom layouts. Build it once, then reuse
it. Choose the kernel belonging to the environment where you installed the package.
Use **Restart Kernel and Run All**. Each run creates a separate evidence directory.

These small examples teach integration, not algorithm performance. All compilation
uses **wave scheduling with local instruction decomposition enabled**. Area means
all cells in the rectangular grid; slices are completed compiler frames; their
product is measured in tile-slices, not physical qubit-cycles. `M` cells are abstract
magic-state supplies, with compiler production timing fixed at 10.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import json
import subprocess

# Start Jupyter from the repository root or its notebooks directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/lattice_floorplan").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository checkout.")

# Change these two paths if you built the adapter somewhere else.
ADAPTER = ROOT / ".deps/adapter/liblsqecc_adapter"
EXPORTER = ROOT / ".deps/adapter/export_layout"
for executable in (ADAPTER, EXPORTER):
    if not executable.is_file():
        raise FileNotFoundError(f"Build the adapter first; missing {executable}")

from lattice_floorplan.compiler import Compiler, CompilerConfig
from lattice_floorplan.geometry import GridState, GeometryOracle
from lattice_floorplan.layout import Layout

# This example is already synthesized into Clifford+T gates.
qasm = (ROOT / "docs/_static/example.qasm").read_text()
compiler = Compiler(ADAPTER, CompilerConfig(local=True, timeout=30))
(ROOT / "runs").mkdir(exist_ok=True)
RUN = Path(mkdtemp(prefix="notebook-", dir=ROOT / "runs"))
print("Evidence directory:", RUN.relative_to(ROOT))


## The connection point

Your algorithm owns candidate generation and selection. The evaluator owns circuit
relabeling, compiler execution, caching and evidence. No subclass or registration
is required: pass a Python callable that scores a `GridState` to your algorithm.

The example below is a deliberately small random search over single-patch moves.
Its restricted proposals are for teaching only; they are not the package's full
geometry search and make no superiority claim. Replace `my_search` with your own
algorithm, including resizing, supply placement or learned proposals as needed.
Candidates must use the supported grid format and preserve explicit logical
identities. Circuit synthesis is a separate step.


In [ ]:
from lattice_floorplan.geometry import single_candidates, valid_geometry, write_layout_output
import random

initial = GridState.parse(subprocess.run(
    [str(EXPORTER), "edpc", "5"], check=True, capture_output=True, text=True
).stdout)
evaluator = GeometryOracle(compiler, RUN / "search-evidence")

def score(grid):
    """Return volume in tile-slices, or None for an invalid/failed layout.

    Args:
        grid (GridState): Candidate tiles and ordered logical-qubit coordinates.

    Returns:
        int | None: Area times completed slices; None means reject the candidate.
    """
    if not valid_geometry(grid):
        return None
    record = evaluator.evaluate(qasm, grid)
    if record["status"] != "success":
        return None
    return grid.area * record["slices"]


def my_search(start, evaluate, requests=12, seed=61):
    """Demonstrate a user algorithm consuming a scoring callable.

    Args:
        start (GridState): Starting layout with explicit logical identities.
        evaluate (Callable): Function taking GridState and returning int or None.
        requests (int): Positive evaluation budget including the initial layout.
        seed (int): Seed for candidate selection.

    Returns:
        tuple[GridState, int, list]: Best grid, its volume, and request history.

    Raises:
        ValueError: The budget is nonpositive or the initial layout fails.
    """
    if requests < 1:
        raise ValueError("requests must be positive")
    rng = random.Random(seed)
    best = start
    best_volume = evaluate(best)
    if best_volume is None:
        raise ValueError("Initial layout failed compilation")
    history = [{"request": 1, "volume": best_volume}]
    for request in range(2, requests + 1):
        candidates = single_candidates(best)
        candidate = rng.choice(candidates)[1] if candidates else best
        volume = evaluate(candidate)
        history.append({"request": request, "volume": volume})
        if volume is not None and volume < best_volume:
            best, best_volume = candidate, volume
    return best, best_volume, history

winner, volume, history = my_search(initial, score)
print("Best volume:", volume)
print("History:", history)
print("Evaluator accounting:", evaluator.accounting())


## Export and verify your algorithm's output

`write_layout_output` saves the bundle but does not validate it by compilation.
Recompile the **mapped** circuit with its exported grid directly through `Compiler`.
This bypasses the search cache and checks the actual exported pair. Unlike the
previous notebook's oracle example, this call must use the mapped circuit because
`Compiler.run` does not perform logical-identity relabeling.


A **bundle** is an ordinary layout output folder, not a special file format. It
contains `layout.txt`, `layout.svg`, `identity.json`, `original-circuit.qasm`,
`mapped-circuit.qasm` and `resources.json`. Keep them together: the grid alone
does not record the logical-qubit identities.


In [ ]:
bundle = RUN / "my-algorithm-winner"
write_layout_output(bundle, winner, qasm, title="Custom algorithm example")
replay = compiler.run(
    (bundle / "mapped-circuit.qasm").read_text(),
    Layout.read(bundle / "layout.txt"), RUN / "winner-replay",
    input_format="qasm", retain_slices=True, preprocess_layout="edpc",
)
if replay["status"] != "success":
    raise RuntimeError(replay)
assert winner.area * replay["slices"] == volume
print("Verified winner slices:", replay["slices"])
print("Exported bundle:", bundle.relative_to(ROOT))


## What to change for your algorithm

Replace `my_search`; keep the `score(GridState)` contract and final export/replay.
If you create a grid from text, pass ordered logical coordinates to
`GridState.parse(text, coordinates)`; `coordinates[i]` identifies logical qubit i.
Without coordinates, parsing assigns identities in row-major order.

`GeometryOracle` always uses EDPC preprocessing and wave scheduling. It does not
reproduce compact preprocessing; use notebook 1 for native generator comparisons.
Use separate evidence roots for separate processes. Declare your search budget,
random seeds and proposal restrictions. A serious method comparison must evaluate
all eight circuit families, report failed compilations, and separate search from
training/tuning cost. This notebook is only an executable integration example.
